# Wakil AI — Phase 3 (Part 1): A Language Model From Scratch

Until now Wakil AI could **read** (recognize digits). Now it learns to **write**.

A **language model** does one thing: given some text, it predicts **what comes next**. ChatGPT and Claude are, at their core, gigantic versions of this idea: they predict the next token again and again.

In this notebook, everything is NumPy and written by hand:
1. **Tokenizer**: text → numbers
2. **Bigram model**: the simplest language model, built by counting
3. **Embeddings**: each character becomes a learned vector
4. **Neural language model**: predicts the next character from the previous 8, trained with the same forward → loss → backward → update loop you know from Phase 1
5. **Text generation** with **temperature**

| | |
|---|---|
| ✅ Implemented from scratch | NumPy only |
| ✅ Trained from scratch | Random initial weights, learns only from the text below |
| ❌ Pretrained model / fine-tuning / AI API | Not used |

**Dataset:** *Tiny Shakespeare*: 1.1 million characters of Shakespeare's plays (public domain).

**Be realistic:** our model has ~50,000 parameters and trains for ~20 seconds on a CPU. It will invent words that *look like* Shakespeare, with character names and dialogue structure, but it doesn't understand anything. Models like Claude have billions of parameters, are trained on trillions of tokens with thousands of GPUs, and cost huge amounts of money and engineering. **The principle is the same; the scale is not.**

**How to run:** Runtime → Run all (about 1–2 minutes on free Colab CPU).

## Cell 1 — Setup

In [ ]:
import json
import os
import time
import urllib.request

import numpy as np
import matplotlib.pyplot as plt

SEED = 42
print("NumPy", np.__version__)

## Cell 2 — Load the text

**Expected output:** `1,115,394 characters` and the beginning of a play (`First Citizen: Before we proceed any further, hear me speak.`).

💡 Want Wakil AI to write in French or Arabic later? Replace this file with any UTF-8 text (a public-domain book from Project Gutenberg, for example). Nothing else changes.

In [ ]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
PATH = "tinyshakespeare.txt"
if not os.path.exists(PATH):
    urllib.request.urlretrieve(URL, PATH)
with open(PATH, encoding="utf-8") as f:
    text = f.read()

print(f"{len(text):,} characters")
print("-" * 40)
print(text[:300])

## Cell 3 — The tokenizer: text → numbers

A neural network only understands numbers. A **tokenizer** cuts text into **tokens** and gives each one an id.

We use the simplest tokenizer: **one token = one character**. The vocabulary is the list of all distinct characters (65 here: letters, digits, punctuation, space, newline).

| Tokenizer type | Example for "unhappy" | Used by |
|---|---|---|
| Characters | `u n h a p p y` (7 tokens) | Wakil AI (now) |
| Words | `unhappy` (1 token) | old models |
| Subwords (BPE) | `un happy` (2 tokens) | GPT, Claude, Llama… |

Subwords are a compromise: a small vocabulary (like characters) but short sequences (like words). Characters are perfect for learning because there's nothing hidden.

**Expected output:** a vocabulary of 65 characters, and `decode(encode("Hello Wakil"))` gives back the same text.

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}   # string -> integer
itos = {i: ch for i, ch in enumerate(chars)}   # integer -> string


def encode(s):
    return np.array([stoi[ch] for ch in s], dtype=np.int64)


def decode(ids):
    return "".join(itos[int(i)] for i in ids)


print("Vocabulary size:", vocab_size)
print("Vocabulary:", repr("".join(chars)))
print("encode('Hello Wakil') =", encode("Hello Wakil"))
print("decode(...)           =", decode(encode("Hello Wakil")))

## Cell 4 — Train / validation split

Same rule as Phase 2: the model learns from **train**, and we judge it on **validation**: text it never saw.

⚠️ This time the split is **contiguous** (first 90% / last 10%), not random. Training examples are overlapping windows of text, so a random split would put almost the same sentences on both sides. The validation score would then be too optimistic.

**Expected output:** `train: 1,003,854 | validation: 111,540`.

In [ ]:
ids = encode(text)
n = int(0.9 * len(ids))
train_ids, val_ids = ids[:n], ids[n:]
print(f"train: {len(train_ids):,} | validation: {len(val_ids):,}")

## Cell 5 — Next-character prediction: what is a training example?

From one window of text we get many examples: **input = the previous characters, target = the next character**.

With a context of 8 characters, the text `First Citizen` gives:

In [ ]:
CONTEXT = 8
window = train_ids[: CONTEXT + 6]
for t in range(CONTEXT, len(window)):
    print(f"{decode(window[t - CONTEXT:t])!r:>14}  ->  {decode(window[t:t + 1])!r}")


def get_batch(data, context, batch_size, rng):
    # Random windows: X = `context` characters, Y = the character that follows each window.
    starts = rng.integers(0, len(data) - context, size=batch_size)
    X = data[starts[:, None] + np.arange(context)]   # (batch_size, context)
    Y = data[starts + context]                       # (batch_size,)
    return X, Y


X, Y = get_batch(train_ids, CONTEXT, 4, np.random.default_rng(0))
print("\nA random batch: X shape", X.shape, "| Y shape", Y.shape)
for x, y in zip(X, Y):
    print(f"{decode(x)!r:>14}  ->  {decode([y])!r}")

## Cell 6 — Model 0: the bigram model (just counting)

The simplest possible language model only looks at the **last character**: for each character, count which characters follow it in the training text.

$$P(\text{next} = b \mid \text{current} = a) = \frac{\text{count}(a \to b) + 1}{\sum_{c} \big(\text{count}(a \to c) + 1\big)}$$

The `+1` (called **smoothing**) makes sure no pair has a probability of exactly 0. Otherwise the loss would be infinite on a pair never seen in training.

### How do we measure a language model?
Same loss as Phase 1: **cross-entropy** = average of $-\log P(\text{correct next character})$. It measures the model's **surprise**.
- Uniform guess among 65 characters: $\ln(65) = 4.17$
- **Perplexity** $= e^{\text{loss}}$ means "the model hesitates as if choosing between N characters". Lower is better.

**Expected output:** validation loss ≈ **2.48** (perplexity ≈ 12) against 4.17 (perplexity 65) for a random guess, then some generated "text". It has the right letter frequencies and short "words", but it's mostly nonsense: one character of memory is not enough.

In [ ]:
counts = np.zeros((vocab_size, vocab_size))
np.add.at(counts, (train_ids[:-1], train_ids[1:]), 1)     # count every pair (a, b)
bigram_probs = (counts + 1) / (counts + 1).sum(axis=1, keepdims=True)


def bigram_loss(data):
    return float(-np.mean(np.log(bigram_probs[data[:-1], data[1:]])))


bigram_val = bigram_loss(val_ids)
print(f"Uniform guess:  loss {np.log(vocab_size):.4f}  (perplexity {vocab_size})")
print(f"Bigram (train): loss {bigram_loss(train_ids):.4f}")
print(f"Bigram (val):   loss {bigram_val:.4f}  (perplexity {np.exp(bigram_val):.1f})")

top = np.argsort(bigram_probs[stoi["q"]])[::-1][:3]
print("\nAfter 'q', the most likely next characters:",
      [(itos[i], f"{bigram_probs[stoi['q'], i]:.0%}") for i in top])

rng = np.random.default_rng(SEED)
out = [stoi["\n"]]
for _ in range(200):
    out.append(rng.choice(vocab_size, p=bigram_probs[out[-1]]))
print("\n--- Bigram model writes ---")
print(decode(out))

## Cell 7 — Embeddings: giving each character a vector

To look at more than one character, we use a neural network. But how do we feed it a character id like `46`? The number 46 has no meaning: "h" is not "bigger" than "a".

**Embedding:** a table `C` with one row of `d` learned numbers per character (here 65 × 16). The character with id 46 becomes the row `C[46]`, a vector of 16 numbers.

- Mathematically, `C[i]` is exactly `one_hot(i) @ C`. It's a normal linear layer, just computed faster with a lookup.
- The rows start random and are **learned by backpropagation**, like every other weight. Characters that behave similarly (vowels, for example) tend to end up with similar vectors.
- **This is exactly how GPT and Claude start**: token id → embedding vector.

### The model (Bengio et al., 2003)
```
8 previous characters (ids)   e.g. "First Ci"
   → embeddings: 8 vectors of 16          E  = C[X]               (B, 8, 16)
   → concatenate: 1 vector of 128         A0 = E.reshape(B, 128)
   → hidden layer: 256 neurons + ReLU     A1 = ReLU(A0 W1 + b1)
   → output: 65 scores → softmax          P  = softmax(A1 W2 + b2)  → P(next character)
```

### The only new piece of backpropagation: the embedding gradient
The gradient flows back to `A0`, gets reshaped to (B, 8, 16), and then each slice is **added** to the row of the character that produced it. If the letter "e" appears 3 times in the batch, its row receives 3 contributions. `np.add.at` does this addition correctly even with repeated ids. A plain `dC[X] += dE` would silently keep only one!

**Expected output:** `Parameters: 50,769` and an initial loss close to ln(65) = 4.17.

In [ ]:
def relu(z):
    return np.maximum(0, z)


def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


class CharMLP:

    def __init__(self, vocab_size, context=8, emb_dim=16, hidden=256, seed=SEED, dtype=np.float32):
        self.config = {"vocab_size": vocab_size, "context": context, "emb_dim": emb_dim, "hidden": hidden}
        self.context = context
        rng = np.random.default_rng(seed)
        fan_in = context * emb_dim
        self.params = {
            "C": rng.standard_normal((vocab_size, emb_dim)).astype(dtype),
            "W1": (rng.standard_normal((fan_in, hidden)) * np.sqrt(2.0 / fan_in)).astype(dtype),
            "b1": np.zeros(hidden, dtype=dtype),
            # small output weights so the first predictions are almost uniform
            "W2": (rng.standard_normal((hidden, vocab_size)) * 0.1 * np.sqrt(1.0 / hidden)).astype(dtype),
            "b2": np.zeros(vocab_size, dtype=dtype),
        }

    def forward(self, X):
        p = self.params
        E = p["C"][X]                       # (B, T, d)  embedding lookup
        A0 = E.reshape(len(X), -1)          # (B, T*d)   concatenate
        Z1 = A0 @ p["W1"] + p["b1"]
        A1 = relu(Z1)
        logits = A1 @ p["W2"] + p["b2"]
        probs = softmax(logits)
        return probs, {"X": X, "A0": A0, "Z1": Z1, "A1": A1, "probs": probs, "logits": logits}

    def backward(self, cache, Y):
        X, A0, Z1, A1, probs = cache["X"], cache["A0"], cache["Z1"], cache["A1"], cache["probs"]
        p, m = self.params, len(Y)
        dlogits = probs.copy()
        dlogits[np.arange(m), Y] -= 1        # P - one_hot(Y), as in Phase 1
        dlogits /= m
        dW2, db2 = A1.T @ dlogits, dlogits.sum(axis=0)
        dZ1 = (dlogits @ p["W2"].T) * (Z1 > 0)
        dW1, db1 = A0.T @ dZ1, dZ1.sum(axis=0)
        dE = (dZ1 @ p["W1"].T).reshape(X.shape + (-1,))   # (B, T, d)
        dC = np.zeros_like(p["C"])
        np.add.at(dC, X, dE)                 # scatter-add into the rows of the characters used
        return {"C": dC, "W1": dW1, "b1": db1, "W2": dW2, "b2": db2}

    def update(self, grads, lr):
        for name in self.params:
            self.params[name] -= lr * grads[name]

    def loss(self, X, Y):
        probs = self.forward(X)[0]
        return float(-np.mean(np.log(probs[np.arange(len(Y)), Y] + 1e-12)))

    def num_parameters(self):
        return sum(v.size for v in self.params.values())


model = CharMLP(vocab_size)
X, Y = get_batch(val_ids, CONTEXT, 1000, np.random.default_rng(0))
print(f"Parameters: {model.num_parameters():,}")
print(f"Initial loss: {model.loss(X, Y):.4f}  (uniform guess = ln(65) = {np.log(vocab_size):.4f})")

## Cell 8 — Gradient check (including the embeddings)

Same idea as Phase 1: compare `backward()` with numerical derivatives on a tiny model. The batch contains **repeated characters on purpose** to test `np.add.at`.

**Expected output:** five small relative errors (< 1e-6) and `Gradient check PASSED`.

In [ ]:
def gradient_check(eps=1e-5):
    tiny = CharMLP(vocab_size=5, context=3, emb_dim=2, hidden=4, seed=1, dtype=np.float64)
    tiny.params["W2"] *= 10
    X = np.array([[0, 1, 1], [2, 2, 2], [4, 0, 3], [1, 1, 0]])
    Y = np.array([3, 0, 2, 1])
    analytic = tiny.backward(tiny.forward(X)[1], Y)
    worst = 0.0
    for name, param in tiny.params.items():
        numeric = np.zeros_like(param)
        for idx in np.ndindex(param.shape):
            original = param[idx]
            param[idx] = original + eps
            plus = tiny.loss(X, Y)
            param[idx] = original - eps
            minus = tiny.loss(X, Y)
            param[idx] = original
            numeric[idx] = (plus - minus) / (2 * eps)
        err = np.linalg.norm(numeric - analytic[name]) / (np.linalg.norm(numeric) + np.linalg.norm(analytic[name]) + 1e-12)
        worst = max(worst, err)
        print(f"{name:>2}: relative error {err:.2e}")
    print("Gradient check PASSED" if worst < 1e-6 else "Gradient check FAILED")
    return worst < 1e-6


gradient_check_passed = gradient_check()

## Cell 9 — Training

The same loop as Phase 1, with two small changes:
- Each step uses a **random batch of 128 windows** (there are a million possible windows, so we don't go through them in "epochs").
- **Learning rate decay:** big steps (0.1) for the first 75% of training, then smaller steps (0.01) to settle into a good spot.

**Expected output** (our reference run; ~20 s on CPU, your last decimals may differ slightly):
```
step   4000/20000 | lr 0.100 | train loss 2.0126 | val loss 2.1029 | 4s
step   8000/20000 | lr 0.100 | train loss 1.8759 | val loss 2.0078 | 8s
step  12000/20000 | lr 0.100 | train loss 1.7805 | val loss 1.9264 | 11s
step  16000/20000 | lr 0.010 | train loss 1.7160 | val loss 1.8705 | 15s
step  20000/20000 | lr 0.010 | train loss 1.6978 | val loss 1.8557 | 19s
```
The validation loss drops from 4.17 (random) past the bigram's 2.48, down to ≈ **1.86** (perplexity ≈ 6.4). Train loss is a bit lower than validation: a little overfitting, which is normal.

In [ ]:
def estimate_loss(model, data, batch_size=1000, batches=10, seed=0):
    rng = np.random.default_rng(seed)
    return float(np.mean([model.loss(*get_batch(data, model.context, batch_size, rng)) for _ in range(batches)]))


STEPS, BATCH_SIZE, LR = 20000, 128, 0.1
model = CharMLP(vocab_size, context=CONTEXT)
rng = np.random.default_rng(SEED)
history = {"step": [], "train_loss": [], "val_loss": [], "batch_loss": []}
start = time.time()

for step in range(1, STEPS + 1):
    lr = LR if step <= 0.75 * STEPS else LR / 10
    X, Y = get_batch(train_ids, CONTEXT, BATCH_SIZE, rng)
    probs, cache = model.forward(X)                              # forward
    history["batch_loss"].append(float(-np.mean(np.log(probs[np.arange(len(Y)), Y] + 1e-12))))  # loss
    grads = model.backward(cache, Y)                             # backward
    model.update(grads, lr)                                      # update

    if step % 4000 == 0:
        train_loss, val_loss = estimate_loss(model, train_ids), estimate_loss(model, val_ids)
        history["step"].append(step); history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
        print(f"step {step:6d}/{STEPS} | lr {lr:.3f} | train loss {train_loss:.4f} "
              f"| val loss {val_loss:.4f} | {time.time() - start:.0f}s")

final_val = history["val_loss"][-1]
print(f"\nValidation loss {final_val:.4f} (perplexity {np.exp(final_val):.1f}) vs bigram {bigram_val:.4f}")

## Cell 10 — Learning curves

In [ ]:
plt.figure(figsize=(9, 4))
batch = np.array(history["batch_loss"])
plt.plot(np.convolve(batch, np.ones(200) / 200, mode="valid"), alpha=0.5, label="batch loss (moving average)")
plt.plot(history["step"], history["train_loss"], "o-", label="train")
plt.plot(history["step"], history["val_loss"], "o-", label="validation")
plt.axhline(np.log(vocab_size), color="red", linestyle=":", label="uniform guess")
plt.axhline(bigram_val, color="gray", linestyle="--", label="bigram (counting)")
plt.xlabel("step"); plt.ylabel("loss"); plt.ylim(1.5, 4.3); plt.legend()
plt.title("Wakil AI language model")
plt.show()

## Cell 11 — Wakil AI writes! (and what temperature does)

**Generation** = repeat: predict the probabilities of the next character → **sample** one at random according to these probabilities → append it → slide the window.

**Temperature** $\tau$ divides the scores before the softmax: $P = \text{softmax}(\text{logits} / \tau)$
- $\tau < 1$: the model becomes more **confident**, so the text is safer but more repetitive.
- $\tau = 1$: the model's real probabilities.
- $\tau > 1$: flatter probabilities, so the text is more surprising, with more mistakes.

ChatGPT and Claude have the same knob.

**Expected output:** three texts in the style of a Shakespeare play, with character names in capitals followed by `:`, line breaks and punctuation. Most words are invented or misspelled: the model only sees 8 characters and has ~50k parameters.

In [ ]:
def generate(model, prompt, length=300, temperature=1.0, seed=None):
    rng = np.random.default_rng(seed)
    ids = [stoi["\n"]] * model.context + list(encode(prompt))
    for _ in range(length):
        logits = model.forward(np.array([ids[-model.context:]]))[1]["logits"][0] / temperature
        probs = softmax(logits[None, :])[0]
        ids.append(int(rng.choice(len(probs), p=probs)))
    return decode(ids[model.context:])


for temperature in [0.5, 0.8, 1.2]:
    print(f"========== temperature {temperature} ==========")
    print(generate(model, "ROMEO:\n", length=250, temperature=temperature, seed=1))
    print()

## Cell 12 — What did the embeddings learn?

Each character is a vector of 16 numbers. We project them to 2D (with PCA, done with NumPy's SVD) to look at them.

**What to look for:** characters that play similar roles tend to get **similar vectors**. Nobody told the model what a vowel or a capital letter is; any grouping comes only from predicting the next character.

**Be honest about what you see:** squeezing 16 dimensions into 2 loses a lot of information, and our model is tiny, so the groups are **blurry**. In our reference run, many lowercase vowels (`a i u`) sit on the left, and many uppercase letters sit towards the top, but there is a lot of overlap. Bigger models trained longer show much clearer structure. (Try Experiment 2 below and compare.)

In [ ]:
C = model.params["C"].astype(np.float64)
C_centered = C - C.mean(axis=0)
_, _, Vt = np.linalg.svd(C_centered, full_matrices=False)
points = C_centered @ Vt[:2].T

plt.figure(figsize=(9, 8))
for i, ch in itos.items():
    label = {"\n": "\\n", " ": "␣"}.get(ch, ch)
    color = "red" if ch in "aeiouAEIOU" else ("blue" if ch.isupper() else ("green" if not ch.isalpha() else "black"))
    plt.text(points[i, 0], points[i, 1], label, fontsize=13, color=color, ha="center", va="center")
plt.xlim(points[:, 0].min() - 0.5, points[:, 0].max() + 0.5)
plt.ylim(points[:, 1].min() - 0.5, points[:, 1].max() + 0.5)
plt.title("Character embeddings (2D projection). red = vowels, blue = uppercase, green = punctuation")
plt.show()

## Cell 13 — Save and download the model

Same file format as the project's `train_lm.py`, so you can use it with `python generate.py --model wakil_lm.npz` on your PC.

In [ ]:
MODEL_PATH = "wakil_lm.npz"
metadata = {"name": "Wakil AI - character language model", "config": model.config, "steps": STEPS,
            "val_loss": final_val, "bigram_val_loss": bigram_val, "pretrained_weights": False}
np.savez_compressed(MODEL_PATH, config=np.array(json.dumps(model.config)), chars=np.array(json.dumps(chars)),
                    metadata=np.array(json.dumps(metadata)), **model.params)
print(f"Saved {MODEL_PATH} ({os.path.getsize(MODEL_PATH) / 1024:.0f} KB)")

try:
    from google.colab import files
    files.download(MODEL_PATH)
except ImportError:
    print("Not in Colab: the file is in", os.path.abspath(MODEL_PATH))

## Cell 14 — Phase 3 (Part 1) checklist

In [ ]:
sample = generate(model, "ROMEO:\n", length=200, temperature=0.8, seed=3)
checks = [
    ("Tokenizer: decode(encode(text)) == text", decode(encode(text[:5000])) == text[:5000]),
    ("Bigram beats a uniform guess", bigram_val < np.log(vocab_size)),
    ("Backpropagation (with embeddings) verified", gradient_check_passed),
    ("Neural model beats the bigram on validation", final_val < bigram_val),
    ("Validation loss < 2.0", final_val < 2.0),
    ("Generates text with dialogue structure (a 'NAME:' line)",
     any(line.strip().endswith(":") and line.strip()[:-1].isupper() for line in sample.splitlines())),
    ("Model saved", os.path.exists(MODEL_PATH)),
]
for description, ok in checks:
    print(("✅ " if ok else "❌ ") + description)
print("\nPHASE 3 PART 1 COMPLETE 🎉" if all(ok for _, ok in checks) else "\nNot complete yet.")

## What's missing, and why we need attention (Part 2)

Our model has real limits:
1. **Fixed, short memory:** it sees exactly 8 characters. It can't know who is speaking or how the sentence started.
2. **No notion of "which characters matter":** each of the 8 positions has its own fixed weights in `W1`. Making the context 100 characters long would mean a huge `W1`, and the model still couldn't *choose* what to look at.

**Self-attention**, the heart of the **Transformer** (the "T" of GPT), fixes this: each position *looks at* all the previous ones and *decides* how much each one matters for predicting the next token, with weights computed from the data itself.

**Part 2:** we build a small Transformer from scratch: attention, positional embeddings and residual connections, then compare it with this model.

## Experiments (try at least one)
1. `CONTEXT = 3` vs `16`: how does the validation loss change?
2. `emb_dim=2`: the embedding plot becomes the real space (no projection). What do you see?
3. Train on another language: replace `tinyshakespeare.txt` with a French or Arabic public-domain book from Project Gutenberg.